In [2]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [7]:
doc = documents[10]
doc

{'content': "# Agents\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=6uG4_Ivv60E&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn Part 1 of this module we built RAG pipelines.\n\nEvery pipeline we wrote followed the same flow:\n\n- search the FAQ,\n- build a prompt with the results,\n- send it to the LLM.\n\nThis returns good answers when the user's query matches the documents.\nThe search finds the right entry, the LLM reads it, and you get a\nhelpful reply.\n\nOften, though, the search returns nothing useful.\n\n- Maybe the user made a typo.\n- Maybe they asked the question in an unusual way.\n- Maybe they need information from two different searches.\n\nWe use lexical search here, so the search looks for an exact match.\nOne typo and it misses the entry it needed. In our pipeline there's\nno recovery. The search runs once, and if it returns garbage the LLM\ngets garbage. Our pipeline always does the same thing, no matter what.\n\nInstead of routing the user question str

In [6]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [4]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [9]:
from dotenv import load_dotenv
from evaluation_utils import llm_structured_retry
from openai import OpenAI
import json
    
load_dotenv()
openai_client = OpenAI()

def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "filename": doc["filename"]
        })

    return results, usage


In [11]:
sample_documents = documents[:3]

In [12]:
from tqdm.auto import tqdm

sample_ground_truth = []
sample_usages = []

for doc in tqdm(sample_documents):
    results, usage = generate_ground_truth(doc)
    sample_ground_truth.extend(results)
    sample_usages.append(usage)
    

  0%|          | 0/3 [00:00<?, ?it/s]

In [13]:
sample_ground_truth

[{'question': 'What problem does retrieval-augmented generation solve for an LLM?',
  'filename': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'Why does the course treat the model like a black box instead of explaining how it works inside?',
  'filename': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'What are the main weaknesses of an LLM that RAG helps with?',
  'filename': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'What is the course building as the example RAG app in this module?',
  'filename': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'What will the first part of the module cover before making the system agentic?',
  'filename': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'What do I need installed before I can follow this module, and do I need anything beyond Python and Jupyter?',
  'filename': '01-agentic-rag/lessons/02-environment.md'},
 {'question': 'How do I set up a new project for the lesson from scratch with uv?',
  'filename':

In [14]:
sample_usages

[ResponseUsage(input_tokens=1020, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=95, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1115),
 ResponseUsage(input_tokens=1286, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=119, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1405),
 ResponseUsage(input_tokens=1753, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=112, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1865)]

In [16]:
import pandas as pd
df_usage = pd.DataFrame(sample_usages)
df_usage

,0,1,2,3,4
0,"(input_tokens, 1020)","(input_tokens_details, InputTokensDetails(cach...","(output_tokens, 95)","(output_tokens_details, OutputTokensDetails(re...","(total_tokens, 1115)"
1,"(input_tokens, 1286)","(input_tokens_details, InputTokensDetails(cach...","(output_tokens, 119)","(output_tokens_details, OutputTokensDetails(re...","(total_tokens, 1405)"
2,"(input_tokens, 1753)","(input_tokens_details, InputTokensDetails(cach...","(output_tokens, 112)","(output_tokens_details, OutputTokensDetails(re...","(total_tokens, 1865)"


In [23]:
from statistics import mean

input_tokens = [usage.input_tokens for usage in sample_usages]
mean(input_tokens)

1353

In [ ]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/cohorts/2026/04-evaluation/ground-truth.csv

--2026-07-12 18:07:35--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/cohorts/2026/04-evaluation/ground-truth.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 48627 (47K) [text/plain]
Saving to: ‘ground-truth.csv’

ground-truth.csv    100%[===================>]  47.49K  --.-KB/s    in 0.002s  

2026-07-12 18:07:35 (25.7 MB/s) - ‘ground-truth.csv’ saved [48627/48627]



In [29]:
df_ground_truth = pd.read_csv("./ground-truth.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [30]:
ground_truth[:5]

[{'question': "What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?",
  'filename': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'Why does this course build the RAG project in plain Python instead of starting with a framework or library?',
  'filename': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'What are the main weaknesses of large language models that this module is trying to work around?',
  'filename': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'What will the course build in the first part of the module, and how is the second part different?',
  'filename': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'What kind of example app are you building here, and what data will it answer questions from?',
  'filename': '01-agentic-rag/lessons/01-intro.md'}]

In [46]:
len(ground_truth)

360

In [32]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [33]:
len(chunks)

295

In [34]:
chunks[0]

{'start': 0,
 'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phon

In [58]:
from minsearch import Index, VectorSearch
from embedder import Embedder

embed = Embedder()
import numpy as np

# Build Text Search
index = Index(
    text_fields=["content"]
)
index.fit(chunks)

# Build Vector Search
matrix = []

for chunk in chunks:
    vector = embed.encode_batch([chunk["content"]])
    matrix.append(vector)

X = np.vstack(matrix)

vector_index = VectorSearch(
    keyword_fields=["content"]
)

vector_index.fit(X, chunks)

In [77]:
def text_search(query, num_results=5):

    return index.search(
        query,
        num_results=num_results
    )

def vector_search(query, num_results=5):

    return vector_index.search(
        embed.encode(query),
        num_results=num_results
    )

In [40]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [78]:
def hybrid_search(query, k=60, num_results=10):
    text_results = text_search(query, num_results=num_results)
    vector_results = vector_search(query, num_results=num_results)
    return rrf([text_results, vector_results], k=k)

In [47]:
q = ground_truth[0]["question"]
q

"What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?"

In [52]:
text_search(q)

[{'start': 3000,
  'content': 'we drop it.\n\nBuild a prompt that includes both the question and the context:\n\n```python\nprompt = f"""\nYour task is to answer questions from the course participants\nbased on the provided context.\n\nUse the context to find relevant information and provide accurate\nanswers. If the answer is not found in the context,\nrespond with "I don\'t know."\n\nQuestion:\n{question}\n\nContext:\n{context}\n"""\n```\n\nInstead of sending the raw question to the LLM, we send this prompt:\n\n```python\nanswer = llm(prompt)\nprint(answer)\n```\n\nAfter that, the answer is correct: "Yes, you can still join. If you want to\nreceive a certificate, you need to submit your project while\nsubmissions are still open."\n\nThis is the answer we actually want to give to our students. What we\njust did is nothing but RAG.\n\n## Retrieval plus generation\n\nRAG stands for Retrieval-Augmented Generation. Generation is the LLM\nproducing text, and retrieval is search. We retriev

In [59]:
vector_search(q)

[{'start': 0,
  'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour ph

In [60]:
def compute_relevance(q, search_function):
    filename = q["filename"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == filename))

    return relevance

In [61]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [62]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

In [63]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)

In [64]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [ ]:
evaluate(
    ground_truth,
    text_search
)

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.7583333333333333, 'mrr': 0.5942592592592594}

In [66]:
evaluate(
    ground_truth,
    vector_search
)

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.725, 'mrr': 0.5486111111111112}

In [80]:
evaluate(
    ground_truth,
    lambda query: hybrid_search(query, k=1)
)

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.8388888888888889, 'mrr': 0.6481944444444449}

In [84]:
evaluate(
    ground_truth,
    lambda query: hybrid_search(query, k=50)
)

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}

In [79]:
evaluate(
    ground_truth,
    lambda query: hybrid_search(query, k=100)
)

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}

In [82]:
evaluate(
    ground_truth,
    lambda query: hybrid_search(query, k=200)
)

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}